# Chapter 11 — Review and customize a schematic

Source candidate · CONVERGING · checkpoint-bound evidence

> **Source candidate / CONVERGING.** Authoring figures are genuine
> exports bound to the corresponding source checkpoints, not evidence
> that every cell or numerical request has executed. The website runs no
> kernels or solvers; the generated Notebook remains zero-output.

Two coupled grounded LC subsystems can be electrically clear yet
visually hard to read. This Chapter reviews the complete literal-value
circuit first, then uses pure presentation hints without changing
physical flow or selecting an analysis View.

## Lesson 11.1 — Author two coupled LC subsystems

### Declare the readout LC subsystem

Start with the root and its readout child so the child owns its native
leaves, parallel relation, and one public `readout_terminal` boundary.

In [ ]:
from pathlib import Path

from scnsim import (
    CircuitDiagramSpec,
    CircuitPlan,
    DiagramAxis,
    DiagramSide,
    SchematicLayout,
    Theme,
    components,
    units as u,
)

plan = CircuitPlan(id="review_structured_schematic")
readout = plan.subsystem(id="readout")
readout_capacitor = readout.add(
    components.capacitor(
        id="capacitor",
        capacitance=110.0 * u.fF,
    )
)
readout_inductor = readout.add(
    components.inductor(
        id="inductor",
        inductance=5.8 * u.nH,
    )
)
readout_terminal_bus = readout.bus(id="terminal")
readout_parallel = readout.parallel(
    id="parallel_lc",
    start=readout_terminal_bus,
    branches=((readout_capacitor,), (readout_inductor,)),
    end=readout.ground,
)
readout_terminal = readout.expose_pin(
    id="terminal",
    at=readout_terminal_bus,
)

The returned terminal and `readout_parallel` handle are the only readout
references used by root wiring and later presentation hints; the LC
values are literal native component values in this lesson.

### Declare the storage LC subsystem

Repeat the independent child declaration for storage; it publishes its
own terminal rather than sharing readout internals.

In [ ]:
storage = plan.subsystem(id="storage")
storage_capacitor = storage.add(
    components.capacitor(
        id="capacitor",
        capacitance=100.0 * u.fF,
    )
)
storage_inductor = storage.add(
    components.inductor(
        id="inductor",
        inductance=6.0 * u.nH,
    )
)
storage_terminal_bus = storage.bus(id="terminal")
storage_parallel = storage.parallel(
    id="parallel_lc",
    start=storage_terminal_bus,
    branches=((storage_capacitor,), (storage_inductor,)),
    end=storage.ground,
)
storage_terminal = storage.expose_pin(
    id="terminal",
    at=storage_terminal_bus,
)

`storage_terminal` and `storage_parallel` now provide the root boundary
and the branch handle consumed by the layout cell.

### Add the root coupling

The 6 fF element stays between separate root buses; the child terminals
are linked only to their respective root nodes.

In [ ]:
readout_node_bus = plan.bus(id="readout_node")
storage_node_bus = plan.bus(id="storage_node")
coupler = plan.add(
    components.capacitor(
        id="readout_storage_coupler",
        capacitance=6.0 * u.fF,
    )
)
coupling = plan.series(
    id="readout_storage",
    start=readout_node_bus,
    elements=(coupler,),
    end=storage_node_bus,
)
plan.link(
    id="readout_child",
    endpoints=(readout_node_bus, readout_terminal),
)
plan.link(
    id="storage_child",
    endpoints=(storage_node_bus, storage_terminal),
)

The root has two separate nodes, the ordered `coupling` relation, and
one link per public child terminal; the next cell promotes their
terminated boundaries.

### Promote the two terminated Ports

In [ ]:
readout_port = plan.add_port(
    id="readout_port",
    at=readout_node_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
storage_port = plan.add_port(
    id="storage_port",
    at=storage_node_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)

`readout` and `storage` are complete inline child regions and
independent root peers: their native leaves are already assembled inside
their own parallel relations. The root’s 6 fF coupler is a separate
structural use, while the direct links bind public boundaries without
adding component slots. The Ports complete the physical declaration and
remain root-bus handles for automatic and hinted presentation.

## Lesson 11.2 — Compare automatic and hinted presentation

### Render the automatic layout

In [ ]:
automatic = plan.render_schematic(
    CircuitDiagramSpec(
        representation="authoring",
        theme=Theme.AUTO,
        show_parameter_values=True,
        show_provenance=True,
        layout=None,
    )
)
automatic.show()

`automatic` records the declaration-order presentation without a layout
hint; keep it to compare with the next, explicitly hinted rendering.

### Construct explicit presentation hints

Scope `order` names only direct peer subsystems and must list every
eligible peer exactly once; omitting a peer is a layout validation
failure, not a partial hint. ParallelRef keys may permute only their
complete `.branches` tuples; electrical series order is immutable. Every
parallel group defaults to vertical paths from top to bottom with
branches left-to-right; a scope axis does not rotate a nested parallel
group.

In [ ]:
layout = SchematicLayout(
    axes={
        readout: DiagramAxis.VERTICAL,
        storage: DiagramAxis.VERTICAL,
        readout_parallel: DiagramAxis.HORIZONTAL,
    },
    order={
        plan: (readout, storage),
        readout_parallel: tuple(reversed(readout_parallel.branches)),
        storage_parallel: storage_parallel.branches,
    },
    terminal_sides={
        readout_terminal: DiagramSide.RIGHT,
        storage_terminal: DiagramSide.LEFT,
    },
    port_sides={
        readout_port: DiagramSide.LEFT,
        storage_port: DiagramSide.RIGHT,
    },
)

`layout` constrains scope axes, complete peer and branch order, terminal
sides, and Port sides. The scope axes arrange peer regions only. The
`readout_parallel` `ParallelRef` has the explicit `HORIZONTAL` axis
hint, so its paths run horizontally and stack top-to-bottom;
`storage_parallel` remains at the vertical default. These hints consume
authored handles without creating topology.

### Render the hinted layout

In [ ]:
hinted = plan.render_schematic(
    CircuitDiagramSpec(
        representation="authoring",
        theme=Theme.AUTO,
        show_parameter_values=True,
        show_provenance=True,
        layout=layout,
    )
)
hinted_drawing = hinted.show()

`hinted` is a second presentation of the same Plan. Review its audit
alongside the automatic drawing before comparing their declaration
identities. Prepare the relative export directory explicitly in a fresh
kernel before saving it.

### Export the hinted drawing

In [ ]:
export_path = Path(
    "workspaces/advanced-course/review_structured_schematic.svg"
)
export_path.parent.mkdir(parents=True, exist_ok=True)
hinted_drawing.save(export_path)

`export_path` names the explicit execution artifact location. The next
cells remain the separate audit and declaration-identity reviews.

### Inspect the drawings’ audit tables

In [ ]:
automatic.audit.show()

In [ ]:
hinted.audit.show()

Each audit table is visible independently; the next cell checks the
three declaration digest classes directly.

### Compare declaration identities

In [ ]:
assert automatic.audit.plan_sha256 == hinted.audit.plan_sha256
assert automatic.audit.connectivity_sha256 == hinted.audit.connectivity_sha256
assert automatic.audit.semantic_sha256 == hinted.audit.semantic_sha256

Matching digests confirm that the layout hints did not alter the Plan,
connectivity, or semantic declaration.

Hints alter only presentation. Scope peers and series run horizontally
by default. Every parallel group instead defaults to top-to-bottom paths
with branches left-to-right in declared order; a `HORIZONTAL` hint on
that `ParallelRef` rotates only that group into horizontal paths stacked
top-to-bottom. A scope-axis hint does not rotate nested parallel groups.
Geometry is not inferred from a main bus, weighted diameter, or physical
flow.

## Lesson 11.3 — Stitch two public terminals

### Stitch public pins in a separate Plan

This separate zero-component conductive stitch contains two real
grounded LC children. Its one shared root bus joins their public
boundaries directly, so it does not short the main Plan’s 6 fF coupling
element.

In [ ]:
stitch_plan = CircuitPlan(id="direct_stitch")
left = stitch_plan.subsystem(id="left")
left_capacitor = left.add(
    components.capacitor(id="capacitor", capacitance=110.0 * u.fF)
)
left_inductor = left.add(
    components.inductor(id="inductor", inductance=5.8 * u.nH)
)
left_terminal_bus = left.bus(id="terminal")
left_parallel = left.parallel(
    id="parallel_lc",
    start=left_terminal_bus,
    branches=((left_capacitor,), (left_inductor,)),
    end=left.ground,
)
left_terminal = left.expose_pin(id="terminal", at=left_terminal_bus)

The left child is nonempty and publishes `left_terminal`; the next cell
builds the equally independent right child for the shared-node stitch.

In [ ]:
right = stitch_plan.subsystem(id="right")
right_capacitor = right.add(
    components.capacitor(id="capacitor", capacitance=100.0 * u.fF)
)
right_inductor = right.add(
    components.inductor(id="inductor", inductance=6.0 * u.nH)
)
right_terminal_bus = right.bus(id="terminal")
right_parallel = right.parallel(
    id="parallel_lc",
    start=right_terminal_bus,
    branches=((right_capacitor,), (right_inductor,)),
    end=right.ground,
)
right_terminal = right.expose_pin(id="terminal", at=right_terminal_bus)

Both public terminals are ready for one N-endpoint direct link to the
root bus.

In [ ]:
stitch_bus = stitch_plan.bus(id="stitched_node")
stitch_plan.link(
    id="children_stitch",
    endpoints=(stitch_bus, left_terminal, right_terminal),
)

`stitch_bus` is one shared electrical net, so this Plan teaches a
zero-component stitch without touching the main coupler.

## Lesson 11.4 — Ground a public return at its parent

### Make the public return boundary explicit

An ordinary child bus named `return` is not intrinsically ground. This
complete module publishes it, and the parent deliberately grounds the
public return pin; the grounding belongs to the assembly rather than
changing the child’s schema.

In [ ]:
return_plan = CircuitPlan(id="external_return")
return_module = return_plan.subsystem(id="module")
module_signal = return_module.bus(id="signal")
module_return = return_module.bus(id="return")
module_capacitor = return_module.add(
    components.capacitor(id="capacitor", capacitance=6.0 * u.fF)
)
return_module.series(
    id="capacitor_path",
    start=module_signal,
    elements=(module_capacitor,),
    end=module_return,
)
module_signal_pin = return_module.expose_pin(id="signal", at=module_signal)
module_return_pin = return_module.expose_pin(id="return", at=module_return)
return_plan.ground_pins(pins=(module_return_pin,))

`ground_pins` grounds the whole currently connected return net, not a
drawing marker. It accepts public pins at this boundary; a parent must
never use the child-private `module_return` bus directly. A raw
`GroundRef` is not a `link` endpoint, and a Port’s own load-to-ground
policy needs no public ground pin.

## Lesson 11.5 — Common independent returns

### Ground distinct returns without inventing a rail

Distinct public return pins may be grounded in one call when their
modules are otherwise independent. This is ordinary common reference,
not an implicit short between the modules’ signal terminals or a new
graphical grouping.

In [ ]:
two_return_plan = CircuitPlan(id="two_public_returns")
left_return = two_return_plan.subsystem(id="left")
right_return = two_return_plan.subsystem(id="right")
left_signal = left_return.bus(id="signal")
left_return_bus = left_return.bus(id="return")
right_signal = right_return.bus(id="signal")
right_return_bus = right_return.bus(id="return")
left_capacitor = left_return.add(
    components.capacitor(id="capacitor", capacitance=6.0 * u.fF)
)
right_capacitor = right_return.add(
    components.capacitor(id="capacitor", capacitance=6.0 * u.fF)
)
left_return.series(
    id="capacitor_path",
    start=left_signal,
    elements=(left_capacitor,),
    end=left_return_bus,
)
right_return.series(
    id="capacitor_path",
    start=right_signal,
    elements=(right_capacitor,),
    end=right_return_bus,
)
left_return_pin = left_return.expose_pin(id="return", at=left_return_bus)
right_return_pin = right_return.expose_pin(id="return", at=right_return_bus)
two_return_plan.ground_pins(pins=(left_return_pin, right_return_pin))

Normal return commoning grounds the two return nets only. Connecting
both ends of either capacitor to one net would instead short that
physical element and is rejected by authoring validation.

## Lesson 11.6 — Correct expected authoring validation

### Read each safety diagnostic before correcting the declaration

The following validation examples are complete small declarations. They
keep the failed mutation visible, preserve its reusable structure id,
and then use a legal public boundary or corrected physical relation. No
cell assumes the current runtime executes these target validations.

In [ ]:
from scnsim import SCNSimValidationError

parent_signal = return_plan.bus(id="parent_signal")
try:
    return_plan.link(
        id="return_attachment",
        endpoints=(parent_signal, module_signal),
    )
except SCNSimValidationError as private_bus_error:
    print(private_bus_error)
    return_plan.link(
        id="return_attachment",
        endpoints=(parent_signal, module_signal_pin),
    )

`module_signal` remains child-private even though this Chapter still has
its Python variable. The retry reuses the rejected id only after
replacing the private bus with the immediate child’s public signal pin.

In [ ]:
conflict_plan = CircuitPlan(id="transitive_bus_conflict")
conflict_child = conflict_plan.subsystem(id="child")
conflict_capacitor = conflict_child.add(
    components.capacitor(id="capacitor", capacitance=110.0 * u.fF)
)
conflict_inductor = conflict_child.add(
    components.inductor(id="inductor", inductance=5.8 * u.nH)
)
conflict_internal = conflict_child.bus(id="terminal")
conflict_child.parallel(
    id="parallel_lc",
    start=conflict_internal,
    branches=((conflict_capacitor,), (conflict_inductor,)),
    end=conflict_child.ground,
)
conflict_pin = conflict_child.expose_pin(id="terminal", at=conflict_internal)
first_bus = conflict_plan.bus(id="first")
second_bus = conflict_plan.bus(id="second")
conflict_coupler = conflict_plan.add(
    components.capacitor(id="coupler", capacitance=6.0 * u.fF)
)
conflict_plan.link(id="first_attachment", endpoints=(first_bus, conflict_pin))
try:
    conflict_plan.link(
        id="second_attachment",
        endpoints=(second_bus, conflict_pin),
    )
except SCNSimValidationError as second_bus_error:
    print(second_bus_error)
    conflict_plan.series(
        id="second_attachment",
        start=second_bus,
        elements=(conflict_coupler,),
        end=first_bus,
    )

The second call would transitively merge two distinct parent buses, so
it is rejected rather than silently creating an anonymous parent short.

In [ ]:
invalid_plan = CircuitPlan(id="invalid_connections")
shorted_bus = invalid_plan.bus(id="shorted")
shorted_capacitor = invalid_plan.add(
    components.capacitor(id="capacitor", capacitance=6.0 * u.fF)
)
try:
    invalid_plan.series(
        id="shorted_capacitor",
        start=shorted_bus,
        elements=(shorted_capacitor,),
        end=shorted_bus,
    )
except SCNSimValidationError as shorted_element_error:
    print(shorted_element_error)
    distinct_bus = invalid_plan.bus(id="distinct")
    invalid_plan.series(
        id="shorted_capacitor",
        start=shorted_bus,
        elements=(shorted_capacitor,),
        end=distinct_bus,
    )

The repaired relation uses the same capacitor between distinct buses.
The next complete Plan isolates the separate rule for a Port-bound
signal net.

In [ ]:
grounded_port_plan = CircuitPlan(id="grounded_port")
port_child = grounded_port_plan.subsystem(id="module")
port_child_signal = port_child.bus(id="signal")
port_child_return = port_child.bus(id="return")
port_child_capacitor = port_child.add(
    components.capacitor(id="capacitor", capacitance=6.0 * u.fF)
)
port_child.series(
    id="port_capacitor",
    start=port_child_signal,
    elements=(port_child_capacitor,),
    end=port_child_return,
)
port_signal_pin = port_child.expose_pin(id="signal", at=port_child_signal)
port_return_pin = port_child.expose_pin(id="return", at=port_child_return)
port_bus = grounded_port_plan.bus(id="port")
grounded_port_plan.link(
    id="port_signal_attachment",
    endpoints=(port_bus, port_signal_pin),
)
grounded_port_plan.add_port(
    id="terminated",
    at=port_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
try:
    grounded_port_plan.ground_pins(pins=(port_signal_pin,))
except SCNSimValidationError as grounded_port_error:
    print(grounded_port_error)
grounded_port_plan.ground_pins(pins=(port_return_pin,))

A Port already carries its declared load-to-ground policy and is not a
public ground pin. `port_signal_pin` is an eligible immediate-child
public `PinRef` on the Port-bound final net, so its rejection
demonstrates the grounded-Port authoring invariant. The other capacitor
terminal remains distinct and ungrounded before that call, so the
earlier rejection is separately the shorted-element case. The legal
correction grounds the return pin, not the signal pin carrying the Port.

In [ ]:
return_diagram = return_plan.render_schematic(CircuitDiagramSpec())
return_diagram.show()

In [ ]:
return_diagram.audit.show()

## Lesson 11.7 — Declare a four-arm illustration

This separate fixed-value circuit illustrates wiring composition; it
does not replace either LC model above. One central Bus joins four
capacitor Series, each ending at its own terminated-Port bus. The labels
A–D identify branches, not physical signal or energy flow. No solver
question is needed.

### Declare the Plan and its five buses

In [ ]:
from scnsim import SchematicComposition

four_arm_plan = CircuitPlan(id="four_arm_composition")
central_bus = four_arm_plan.bus(id="central")
bus_a = four_arm_plan.bus(id="a")
bus_b = four_arm_plan.bus(id="b")
bus_c = four_arm_plan.bus(id="c")
bus_d = four_arm_plan.bus(id="d")

### Register four real capacitors

In [ ]:
capacitor_a = four_arm_plan.add(
    components.capacitor(id="capacitor_a", capacitance=4.0 * u.fF)
)
capacitor_b = four_arm_plan.add(
    components.capacitor(id="capacitor_b", capacitance=5.0 * u.fF)
)
capacitor_c = four_arm_plan.add(
    components.capacitor(id="capacitor_c", capacitance=6.0 * u.fF)
)
capacitor_d = four_arm_plan.add(
    components.capacitor(id="capacitor_d", capacitance=7.0 * u.fF)
)

### Bind each capacitor to a distinct outer bus

In [ ]:
arm_a = four_arm_plan.series(
    id="arm_a", start=central_bus, elements=(capacitor_a,), end=bus_a,
)
arm_b = four_arm_plan.series(
    id="arm_b", start=central_bus, elements=(capacitor_b,), end=bus_b,
)
arm_c = four_arm_plan.series(
    id="arm_c", start=central_bus, elements=(capacitor_c,), end=bus_c,
)
arm_d = four_arm_plan.series(
    id="arm_d", start=central_bus, elements=(capacitor_d,), end=bus_d,
)

### Complete the four terminated boundaries

In [ ]:
port_a = four_arm_plan.add_port(
    id="port_a", at=bus_a, role="terminated",
    reference_impedance=50.0 * u.ohm,
)
port_b = four_arm_plan.add_port(
    id="port_b", at=bus_b, role="terminated",
    reference_impedance=50.0 * u.ohm,
)
port_c = four_arm_plan.add_port(
    id="port_c", at=bus_c, role="terminated",
    reference_impedance=50.0 * u.ohm,
)
port_d = four_arm_plan.add_port(
    id="port_d", at=bus_d, role="terminated",
    reference_impedance=50.0 * u.ohm,
)

Each Port retains its declared load/reference. The four central Series
boundaries are four presentation attachments on one existing node; the
central Bus is not a fifth arm.

### Begin with fixed Default completion

In [ ]:
four_arm_composition = SchematicComposition.automatic(plan=four_arm_plan)

In [ ]:
four_arm_composition.show()

This read-only inspection distinguishes explicit settings from omitted
fields that receive Default values. It is not an audited drawing. The
constructor, `automatic()`, and a missing layout use the same fixed
completion and public builders, not different rendering engines or a
cost-ranked search. The next cell measures the actual point and labels,
places each body once, and constructs one routing realization.

In [ ]:
four_arm_automatic = four_arm_plan.render_schematic(
    CircuitDiagramSpec(layout=four_arm_composition)
)

In [ ]:
four_arm_automatic.show()

In [ ]:
four_arm_automatic.audit.show()

## Lesson 11.8 — Request one visible Cross

### Choose Block axes and complete Port orientations

These settings describe presentation only. A horizontal or vertical axis
does not reverse the ordered endpoints authored above.

In [ ]:
four_arm_composition.axis(arm_a, DiagramAxis.HORIZONTAL)
four_arm_composition.axis(arm_b, DiagramAxis.HORIZONTAL)
four_arm_composition.axis(arm_c, DiagramAxis.VERTICAL)
four_arm_composition.axis(arm_d, DiagramAxis.VERTICAL)
four_arm_composition.port_orientation(
    port_a, boundary_side=DiagramSide.LEFT, load_side=DiagramSide.BOTTOM,
)
four_arm_composition.port_orientation(
    port_b, boundary_side=DiagramSide.RIGHT, load_side=DiagramSide.BOTTOM,
)
four_arm_composition.port_orientation(
    port_c, boundary_side=DiagramSide.TOP, load_side=DiagramSide.RIGHT,
)
four_arm_composition.port_orientation(
    port_d, boundary_side=DiagramSide.BOTTOM, load_side=DiagramSide.RIGHT,
)

Each Port has two perpendicular directions in the root frame: its hollow
marker side and the direction from its T toward its load/ground. Four
marker sides times two perpendicular load sides give eight poses. The
complete setter replaces both atomically. `port_side()` changes only the
marker boundary while preserving an explicit load side; an incompatible
change rejects without a partial update. `ground_side()` cannot target a
Port. Wiring attaches to the Port’s circuit-facing lead end, not its
marker or internal T.

### Replace only the central wiring group

In [ ]:
central_wiring = four_arm_composition.replace_wiring(at=central_bus)

In [ ]:
central_cross = central_wiring.cross(id="central_cross")

In [ ]:
central_wiring.connect(
    four_arm_composition.endpoint(arm_a, boundary="start"), central_cross.left,
)
central_wiring.connect(
    four_arm_composition.endpoint(arm_b, boundary="start"), central_cross.right,
)
central_wiring.connect(
    four_arm_composition.endpoint(arm_c, boundary="start"), central_cross.top,
)
central_wiring.connect(
    four_arm_composition.endpoint(arm_d, boundary="start"), central_cross.bottom,
)

The explicit structure boundaries distinguish all four contacts. Every
contact and Cross arm is used once. A graphical junction adds no
electrical net, Tap, or Coordinate. When a Tap aliases several actual
contacts, select the intended structure boundary explicitly rather than
guessing an unused contact.

### Inspect, render, and audit separately

In [ ]:
four_arm_composition.show()

In [ ]:
four_arm_custom = four_arm_plan.render_schematic(
    CircuitDiagramSpec(layout=four_arm_composition)
)

In [ ]:
four_arm_custom.show()

In [ ]:
four_arm_custom.audit.show()

Layout verification must find one actual four-arm Cross. Two Ts can
preserve electrical witness A yet violate that requested shape; A/B
remain checks of visible electrical and ownership evidence, not hidden
recipe metadata. These TARGET cells do not claim a runtime-verified
gallery from notebook generation.

A new `SchematicComposition(plan=four_arm_plan)` permits partial
settings: omitted fields use stable defaults, while explicit choices
win. There is no whole-manual completeness requirement. Replacing a
wiring group still means specifying every attachment and arm in that
group; Default never guesses missing manual wires. The caller’s settings
and refs remain unchanged on render success or failure. Unsupported
fixed geometry reports the affected scope, Block, junction, or
constraint without fallback search or partial success.

## Lesson 11.9 — Reuse the final recipe, not the pixels

### Inspect the detached completed composition

In [ ]:
four_arm_custom.composition.show()

Unlike inspection before rendering, this describes the completed public
choices bound to the same verified scene. It is immutable and detached:
inspection still works after the live Plan changes, without exposing
private editable contents or live handles. Compiled diagrams instead
have `composition is None`.

### Create a new fixed editable recipe

In [ ]:
fixed_four_arm = four_arm_custom.composition.to_composition(plan=four_arm_plan)

In [ ]:
fixed_four_arm.show()

The exact captured Plan identity must match. All Default-completed
public choices are now fixed, while hidden internals remain
nonaddressable. The new recipe is independent of both the original
intent and the captured diagram.

In [ ]:
fixed_four_arm_diagram = four_arm_plan.render_schematic(
    CircuitDiagramSpec(layout=fixed_four_arm)
)

In [ ]:
fixed_four_arm_diagram.show()

In [ ]:
fixed_four_arm_diagram.audit.show()

Fixed reuse is not a pixel freeze. Rendering remeasures labels and
geometry and reroutes at the selected single parameter point; an
incompatible point may fail. This fixed-value illustration does not add
a parameter or numerical request to demonstrate that distinction.

There is no global reset-to-auto operation. A fresh partial constructor
can change one field while leaving all other fields at Default, without
changing the saved diagram:

In [ ]:
fresh_four_arm = SchematicComposition(plan=four_arm_plan)
fresh_four_arm.port_side(port_a, DiagramSide.LEFT)

In [ ]:
fresh_four_arm.show()

No complete manual axis/order/wiring setup is needed for that single
override. Its Port side is not reselected to avoid obstacles; other
omitted fields follow the same stable rules used by `automatic()` and
`layout=None`.

In [ ]:
partial_default_diagram = four_arm_plan.render_schematic(
    CircuitDiagramSpec(layout=fresh_four_arm)
)

In [ ]:
partial_default_diagram.show()

In [ ]:
partial_default_diagram.audit.show()

[Previous](10_use_custom_component.qmd) ·
[Next](12_model_n_trace_line.qmd)